# Download SCimilarity Pre-trained Model

One-time setup utility to download the [SCimilarity](https://github.com/Genentech/scimilarity) pre-trained cell annotation model from Zenodo and stage it to a Unity Catalog Volume.

**SCimilarity** (Genentech) is a foundation model for single-cell RNA-seq that:
- Generates cell embeddings using a pre-trained neural network
- Annotates cell types via kNN search against a **~23M cell reference atlas**
- Provides distance-based confidence scores

**After running this notebook**, use the output model path in the Genesis Workbench scanpy pipeline:
- Streamlit form → Advanced Pipeline Options → Annotation Method: `scimilarity` → Model Path: `<output path>`
- Or set the `annotation_model` widget directly in `analyze_single_h5ad`

In [0]:
%pip install scimilarity requests
%restart_python

In [0]:
# Configure the destination UC Volume path
# Update these to match your environment

dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("volume", "singlecell_data", "Volume Name")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/scimilarity_model"
print(f"Model will be downloaded to: {MODEL_DIR}")

In [0]:
import requests
import json

# SCimilarity model is hosted on Zenodo
# Record 10685499 = SCimilarity v1.1 (annotation model + reference atlas)
ZENODO_RECORD_ID = "10685499"

print(f"Querying Zenodo record {ZENODO_RECORD_ID}...")
resp = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}")
resp.raise_for_status()
record = resp.json()

print(f"\nTitle: {record['metadata']['title']}")
print(f"DOI: {record['doi']}")
print(f"Published: {record['metadata']['publication_date']}")
print(f"\nAvailable files:")

files_info = []
for f in record['files']:
    size_mb = f['size'] / (1024 * 1024)
    size_gb = size_mb / 1024
    print(f"  {f['key']:50s}  {size_gb:.2f} GB  ({f['checksum']})")
    files_info.append({
        'key': f['key'],
        'url': f['links']['self'],
        'size_mb': size_mb,
        'checksum': f['checksum']
    })

# Find the model tarball (largest tar.gz file)
tar_files = [f for f in files_info if f['key'].endswith('.tar.gz')]
if tar_files:
    model_file = max(tar_files, key=lambda x: x['size_mb'])
    print(f"\nSelected model archive: {model_file['key']} ({model_file['size_mb']/1024:.2f} GB)")
    DOWNLOAD_URL = model_file['url']
    MODEL_FILENAME = model_file['key']
    EXPECTED_CHECKSUM = model_file['checksum']
else:
    # Fallback: try zip files
    zip_files = [f for f in files_info if f['key'].endswith('.zip')]
    if zip_files:
        model_file = max(zip_files, key=lambda x: x['size_mb'])
        print(f"\nSelected model archive: {model_file['key']} ({model_file['size_mb']/1024:.2f} GB)")
        DOWNLOAD_URL = model_file['url']
        MODEL_FILENAME = model_file['key']
        EXPECTED_CHECKSUM = model_file['checksum']
    else:
        raise RuntimeError(f"No .tar.gz or .zip model archive found in Zenodo record {ZENODO_RECORD_ID}")

In [0]:
import os
import hashlib
import time

os.makedirs(MODEL_DIR, exist_ok=True)
archive_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

# Check if already downloaded
if os.path.exists(archive_path):
    existing_size_mb = os.path.getsize(archive_path) / (1024 * 1024)
    print(f"Archive already exists: {archive_path} ({existing_size_mb:.1f} MB)")
    print("Skipping download. Delete the file to re-download.")
else:
    print(f"Downloading {MODEL_FILENAME} ({model_file['size_mb']/1024:.2f} GB)...")
    print(f"From: {DOWNLOAD_URL}")
    print(f"To:   {archive_path}")
    print()
    
    t0 = time.time()
    
    # Stream download with progress
    response = requests.get(DOWNLOAD_URL, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    downloaded = 0
    chunk_size = 8 * 1024 * 1024  # 8 MB chunks
    
    with open(archive_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                pct = 100.0 * downloaded / total_size
                speed_mbps = (downloaded / (1024*1024)) / max(time.time() - t0, 0.1)
                print(f"\r  {pct:.1f}% ({downloaded/(1024*1024):.0f} MB / {total_size/(1024*1024):.0f} MB) @ {speed_mbps:.1f} MB/s", end='', flush=True)
    
    elapsed = time.time() - t0
    print(f"\n\nDownload complete in {elapsed:.0f}s ({downloaded/(1024*1024):.0f} MB)")

# Verify checksum
print("\nVerifying checksum...")
md5_hash = hashlib.md5()
with open(archive_path, 'rb') as f:
    for chunk in iter(lambda: f.read(8192 * 1024), b''):
        md5_hash.update(chunk)
computed = f"md5:{md5_hash.hexdigest()}"
if computed == EXPECTED_CHECKSUM:
    print(f"  Checksum verified: {computed}")
else:
    print(f"  WARNING: Checksum mismatch!")
    print(f"  Expected: {EXPECTED_CHECKSUM}")
    print(f"  Got:      {computed}")

In [0]:
import tarfile
import zipfile

extracted_dir = os.path.join(MODEL_DIR, "extracted")

if os.path.exists(extracted_dir) and os.listdir(extracted_dir):
    print(f"Model already extracted at: {extracted_dir}")
    print(f"Contents: {os.listdir(extracted_dir)}")
else:
    print(f"Extracting {MODEL_FILENAME}...")
    os.makedirs(extracted_dir, exist_ok=True)
    
    t0 = time.time()
    
    if MODEL_FILENAME.endswith('.tar.gz') or MODEL_FILENAME.endswith('.tgz'):
        with tarfile.open(archive_path, 'r:gz') as tar:
            tar.extractall(path=extracted_dir)
    elif MODEL_FILENAME.endswith('.zip'):
        with zipfile.ZipFile(archive_path, 'r') as z:
            z.extractall(path=extracted_dir)
    else:
        raise RuntimeError(f"Unknown archive format: {MODEL_FILENAME}")
    
    elapsed = time.time() - t0
    print(f"Extraction complete in {elapsed:.0f}s")

# Find the actual model root (may be nested in a subdirectory)
print(f"\nExtracted contents:")
for root, dirs, files in os.walk(extracted_dir):
    level = root.replace(extracted_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Only show 2 levels deep
        for f in files[:10]:
            print(f"{indent}  {f}")
        if len(files) > 10:
            print(f"{indent}  ... and {len(files)-10} more files")

In [0]:
# Find the actual model root — look for gene_order.tsv or encoder/ directory
import glob

model_path = None

# Strategy 1: Look for gene_order.tsv (key SCimilarity model file)
gene_order_files = glob.glob(os.path.join(extracted_dir, '**', 'gene_order.tsv'), recursive=True)
if gene_order_files:
    model_path = os.path.dirname(gene_order_files[0])
    print(f"Found gene_order.tsv at: {gene_order_files[0]}")
    print(f"Model root: {model_path}")

# Strategy 2: Look for model_params.json or similar config
if not model_path:
    config_files = glob.glob(os.path.join(extracted_dir, '**', 'model_params.json'), recursive=True)
    if config_files:
        model_path = os.path.dirname(config_files[0])
        print(f"Found model_params.json at: {config_files[0]}")
        print(f"Model root: {model_path}")

# Strategy 3: Look for any .pt or .pth (PyTorch) files
if not model_path:
    pt_files = glob.glob(os.path.join(extracted_dir, '**', '*.pt'), recursive=True)
    pt_files += glob.glob(os.path.join(extracted_dir, '**', '*.pth'), recursive=True)
    if pt_files:
        # Go up to find the model root
        model_path = os.path.dirname(pt_files[0])
        # If nested (e.g., encoder/model.pt), go up one more
        if os.path.basename(model_path) in ('encoder', 'weights', 'checkpoint'):
            model_path = os.path.dirname(model_path)
        print(f"Found model weights at: {pt_files[0]}")
        print(f"Model root: {model_path}")

# Strategy 4: If extracted_dir has exactly one subdirectory, use it
if not model_path:
    subdirs = [d for d in os.listdir(extracted_dir) if os.path.isdir(os.path.join(extracted_dir, d))]
    if len(subdirs) == 1:
        model_path = os.path.join(extracted_dir, subdirs[0])
        print(f"Using single subdirectory as model root: {model_path}")
    else:
        model_path = extracted_dir
        print(f"Using extraction directory as model root: {model_path}")

print(f"\n{'='*60}")
print(f"MODEL PATH FOR PIPELINE:")
print(f"  {model_path}")
print(f"{'='*60}")
print(f"\nContents:")
for item in sorted(os.listdir(model_path)):
    full = os.path.join(model_path, item)
    if os.path.isdir(full):
        n_files = len(os.listdir(full))
        print(f"  {item}/  ({n_files} files)")
    else:
        size_mb = os.path.getsize(full) / (1024*1024)
        print(f"  {item}  ({size_mb:.1f} MB)")

In [0]:
# Verify SCimilarity can load the model
try:
    from scimilarity.cell_annotation import CellAnnotation
    
    print(f"Loading SCimilarity model from: {model_path}")
    ca = CellAnnotation(model_path=model_path)
    
    print(f"  Model loaded successfully!")
    print(f"  Gene order: {len(ca.gene_order)} genes")
    print(f"  First 5 genes: {ca.gene_order[:5]}")
    print(f"  Last 5 genes: {ca.gene_order[-5:]}")
    
    print(f"\n{'='*60}")
    print(f"SUCCESS — Model is ready for use.")
    print(f"\nUse this path in the scanpy pipeline:")
    print(f"  annotation_method = scimilarity")
    print(f"  annotation_model  = {model_path}")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"Warning: Could not verify model load: {e}")
    print(f"\nThe model files are at: {model_path}")
    print(f"You may need to adjust the path if the directory structure differs.")
    print(f"\nExpected structure:")
    print(f"  {model_path}/")
    print(f"    gene_order.tsv")
    print(f"    encoder/")
    print(f"    reference/ (or knn/)")

In [0]:
# Optionally remove the archive to save Volume storage
# The extracted model files are what's needed; the archive is ~2-3 GB redundant

# Uncomment to delete the archive:
# os.remove(archive_path)
# print(f"Deleted archive: {archive_path}")

print(f"Archive:   {archive_path} ({os.path.getsize(archive_path)/(1024*1024*1024):.2f} GB)")
print(f"Model dir: {model_path}")
print(f"\nTo save space, uncomment the os.remove() line above to delete the archive.")